In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from matplotlib.path import Path

import imageio.v3 as iio

In [2]:
class Environment:

    def __init__(self):

        self.width = 1000
        self.height = 1000

        self.aura_sigma = 50.0

        # Static background
        self.background = torch.zeros(
            (self.height, self.width, 3),
            dtype=torch.float32
        )

        self.add_gaussians()

        # Current world state
        self.field = self.background.clone()

    def add_gaussians(self):

        y, x = torch.meshgrid(
            torch.arange(self.height, dtype=torch.float32),
            torch.arange(self.width, dtype=torch.float32),
            indexing="ij"
        )

        for _ in range(4):

            cx = torch.randint(0, self.width, (1,)).item()
            cy = torch.randint(0, self.height, (1,)).item()
            sigma = torch.empty(1).uniform_(50.0, 150.0).item()

            gaussian = torch.exp(
                -((x - cx) ** 2 + (y - cy) ** 2) /
                (2 * sigma ** 2)
            )

            self.background[:, :, 1] += gaussian

        self.background[:, :, 1].clamp_(0.0, 1.0)

    def update(self, agents):

        # Restore static world
        self.field.copy_(self.background)

        # Draw aura first, then agent body
        for agent in agents:
            self.draw_aura(agent)

        for agent in agents:
            self.draw_agent(agent)

    def draw_aura(self, agent):

        radius = int(self.aura_sigma * 3)

        xmin = max(int(agent.x) - radius, 0)
        xmax = min(int(agent.x) + radius + 1, self.width)

        ymin = max(int(agent.y) - radius, 0)
        ymax = min(int(agent.y) + radius + 1, self.height)

        if xmin >= xmax or ymin >= ymax:
            return

        yy, xx = torch.meshgrid(
            torch.arange(ymin, ymax, dtype=torch.float32),
            torch.arange(xmin, xmax, dtype=torch.float32),
            indexing="ij"
        )

        gaussian = 0.5*torch.exp(
            -((xx - agent.x) ** 2 + (yy - agent.y) ** 2) /
            (2 * self.aura_sigma ** 2)
        )

        color = torch.tensor(agent.color, dtype=torch.float32)

        self.field[ymin:ymax, xmin:xmax] += (
            gaussian.unsqueeze(-1) * color
        )

        self.field[ymin:ymax, xmin:xmax].clamp_(0.0, 1.0)

    def draw_agent(self, agent):

        polygon = agent.get_polygon().numpy()

        xmin = max(int(polygon[:, 0].min()) - 1, 0)
        xmax = min(int(polygon[:, 0].max()) + 2, self.width)

        ymin = max(int(polygon[:, 1].min()) - 1, 0)
        ymax = min(int(polygon[:, 1].max()) + 2, self.height)

        if xmin >= xmax or ymin >= ymax:
            return

        yy, xx = torch.meshgrid(
            torch.arange(ymin, ymax),
            torch.arange(xmin, xmax),
            indexing="ij"
        )

        points = torch.stack(
            (xx.flatten(), yy.flatten()),
            dim=1
        ).numpy()

        mask = Path(polygon).contains_points(points)
        mask = torch.from_numpy(
            mask.reshape(ymax - ymin, xmax - xmin)
        )

        self.field[ymin:ymax, xmin:xmax][mask] = torch.tensor(
            agent.color,
            dtype=torch.float32
        )

    def consume_food(self, agent):

        x0 = max(int(agent.x) - 5, 0)
        x1 = min(int(agent.x) + 5, self.width)

        y0 = max(int(agent.y) - 5, 0)
        y1 = min(int(agent.y) + 5, self.height)

        food = self.background[y0:y1, x0:x1, 1]

        amount = food.sum().item()

        agent.energy += amount

        food.zero_()

In [4]:
class Brain(nn.Module):

    def __init__(self, genome,
                 input_size=300,
                 hidden_size=128,
                 output_size=10):

        super().__init__()

        self.Win, self.Wh, self.Wout = genome.extract_weights()

        self.h = torch.zeros(hidden_size)

        self.leak = 0.6
        self.hidden_max = 10000
        self.add_hidden =  0.5
        self.add_sens =  0.01
        self.output_gain = 100

    def forward(self, x):

        # Sensory input
        sensory = self.Win @ x

        # Clamp sensory drive
        sensory = torch.clamp(sensory, 0, self.hidden_max)

        # Recurrent drive
        recurrent = (self.Wh) @ self.h

        # Leak hidden state
        self.h = (1.0 - self.leak) * self.h

        # Add sensory and recurrent contributions
        self.h = (
            self.h
            + self.add_sens * sensory
            + self.add_hidden * recurrent
        )

        # Clamp hidden activity
        self.h = torch.clamp(self.h, 0, self.hidden_max)

        # Output
        output = self.Wout @ self.h

        # Positive motor neurons
        output = torch.clamp(output, min=0)*self.output_gain

        return output



In [6]:
class Agent:

    def __init__(self, genome):

        self.x = 0.0
        self.y = 0.0

        self.vx = 0.0
        self.vy = 0.0

        self.theta = 0.0
        self.omega = 0.0

        #self.h = torch.zeros(16)

        self.color = [1.0, 0.0, 0.0]

        self.energy = 0.0

        self.brain = Brain(genome)


    def get_input(self, env):

        # Sensor center in the agent's local coordinates
        sensor_center = torch.tensor([-10.0, 0.0])

        c = torch.cos(torch.tensor(self.theta))
        s = torch.sin(torch.tensor(self.theta))

        R = torch.tensor([
            [c, -s],
            [s,  c]
        ])

        sensor_center = R @ sensor_center
        sensor_center[0] += self.x
        sensor_center[1] += self.y

        inputs = torch.zeros(10, 10, 3)

        for i in range(10):
            for j in range(10):

                # Pixel coordinates relative to the sensor center
                local = torch.tensor([
                    i - 4.5,
                    j - 4.5
                ])

                world = R @ local
                world += sensor_center

                x = int(round(world[0].item()))
                y = int(round(world[1].item()))

                if 0 <= x < env.width and 0 <= y < env.height:
                    inputs[j, i] = env.field[y, x]

        return inputs.flatten()

    def get_polygon(self):

        # Rectangle centered at the origin
        corners = torch.tensor([
            [-5.0, -2.5],
            [ 5.0, -2.5],
            [ 5.0,  2.5],
            [-5.0,  2.5],
        ])

        c = torch.cos(torch.tensor(self.theta))
        s = torch.sin(torch.tensor(self.theta))

        R = torch.tensor([
            [c, -s],
            [s,  c]
        ])

        corners = corners @ R.T

        corners[:, 0] += self.x
        corners[:, 1] += self.y

        return corners

    def step(self, env, dt):

        # Sense
        inp = self.get_input(env)

        # Think
        output = self.brain(inp)

        # Motor neurons (non-negative firing)
        forward = output[0] - output[1]
        turn = output[2] - output[3]

        # Act
        # Turn

        self.omega += turn.item() * dt

        self.omega *= 0.1      # angular damping

        #self.theta += self.omega * dt
        self.theta += turn.item() * dt

        # Acceleration
        ax = torch.cos(torch.tensor(self.theta)) * forward
        ay = torch.sin(torch.tensor(self.theta)) * forward

        self.vx += ax.item() * dt
        self.vy += ay.item() * dt

        # Friction / drag
        friction = 0.9
        self.vx *= friction
        self.vy *= friction

        # Position
        self.x += self.vx * dt
        self.y += self.vy * dt


                # Left/right boundaries
        if self.x < 0:
            self.x = 0
            self.vx = 0.0

        elif self.x > env.width - 1:
            self.x = env.width - 1
            self.vx = 0.0

        # Top/bottom boundaries
        if self.y < 0:
            self.y = 0
            self.vy = 0.0

        elif self.y > env.height - 1:
            self.y = env.height - 1
            self.vy = 0.0

        env.consume_food(self)



In [7]:
R_IN  = 18    # 128x300
R_H   = 13    # 128x128
R_OUT = 2     # 10x128

NUM_PARAMETERS_PE = (
    128 * R_IN + R_IN * 300 +       # Ain, Bin
    128 * R_H  + R_H * 128 +        # Ah, Bh
    10 * R_OUT + R_OUT * 128       # Aout, Bout
)

In [8]:
class GenomePE:

    def __init__(self):

        self.chromosome = torch.randn(NUM_PARAMETERS_PE) * 0.1

    def clone(self):

        g = GenomePE()
        g.chromosome = self.chromosome.clone()

        return g

    def mutate(self, mutation_rate=0.1, mutation_scale=0.05):

        mask = torch.rand_like(self.chromosome) < mutation_rate

        self.chromosome += (
            mask
            * torch.randn_like(self.chromosome)
            * mutation_scale
        )

        self.chromosome = torch.clamp(
            self.chromosome,
            -1.0,
            1.0
        )

    def extract_weights(self):

        chromosome = self.chromosome
        idx = 0

        n_Ain = 128 * R_IN
        n_Bin = R_IN * 300
        n_Ah = 128 * R_H
        n_Bh = R_H * 128
        n_Aout = 10 * R_OUT
        n_Bout = R_OUT * 128

        Ain = chromosome[idx:idx+n_Ain].reshape(128, R_IN)
        idx += n_Ain

        Bin = chromosome[idx:idx+n_Bin].reshape(R_IN, 300)
        idx += n_Bin

        Ah = chromosome[idx:idx+n_Ah].reshape(128, R_H)
        idx += n_Ah

        Bh = chromosome[idx:idx+n_Bh].reshape(R_H, 128)
        idx += n_Bh

        Aout = chromosome[idx:idx+n_Aout].reshape(10, R_OUT)
        idx += n_Aout

        Bout = chromosome[idx:idx+n_Bout].reshape(R_OUT, 128)

        Win = Ain @ Bin
        Wh = Ah @ Bh
        Wout = Aout @ Bout

        return Win, Wh, Wout

In [9]:
genome = GenomePE()

Win, Wh, Wout = genome.extract_weights()

print("Win:", Win.shape)
print("Wh:", Wh.shape)
print("Wout:", Wout.shape)
print("chrom_len", len(genome.chromosome))

Win: torch.Size([128, 300])
Wh: torch.Size([128, 128])
Wout: torch.Size([10, 128])
chrom_len 11308


In [43]:
env = Environment()

genomes = []
agents = []

for _ in range(20):

    genome = GenomePE()
    agent = Agent(genome)

    agent.x = torch.randint(0, env.width, (1,)).item()
    agent.y = torch.randint(0, env.height, (1,)).item()
    agent.theta = 2 * torch.pi * torch.rand(1).item()

    genomes.append(genome)
    agents.append(agent)

In [44]:
len(genome.chromosome)

11308

In [ ]:
import imageio.v3 as iio

iio.imwrite("simulation1.mp4", frames, fps=20)

In [ ]:
del frames

In [10]:
NUM_AGENTS = 20

genomes = [GenomePE() for _ in range(NUM_AGENTS)]

In [11]:
def run_generation(genomes, steps=500):

    env = Environment()

    agents = []

    for genome in genomes:

        agent = Agent(genome)

        agent.x = torch.randint(0, env.width, (1,)).item()
        agent.y = torch.randint(0, env.height, (1,)).item()
        agent.theta = 2 * torch.pi * torch.rand(1).item()

        agents.append(agent)


    for t in range(steps):

        env.update(agents)

        for agent in agents:
            agent.step(env, dt=1.0)


    fitness = [
        agent.energy
        for agent in agents
    ]

    return fitness

In [13]:
def reproduce(survivors):

    offspring_counts = [5, 5, 4, 3, 3]

    new_genomes = []

    for parent, count in zip(survivors, offspring_counts):

        for i in range(count):

            child = parent.clone()

            # Keep first offspring identical, mutate the rest
            if i > 0:
                child.mutate()

            new_genomes.append(child)

    return new_genomes

In [66]:
hall_of_fame = []

for g in range(200):

    fitness = run_generation(genomes)

    ranking = sorted(
        range(len(genomes)),
        key=lambda i: fitness[i],
        reverse=True
    )

    # Save best ever with fitness attached
    hall_of_fame.extend([
        (fitness[i], genomes[i].clone())
        for i in ranking[:2]
    ])

    hall_of_fame.sort(key=lambda x: x[0], reverse=True)
    hall_of_fame = hall_of_fame[:20]

    survivors = [genomes[i] for i in ranking[:5]]

    genomes = reproduce(survivors)

    # Inject elites
    for i, (_, genome) in enumerate(hall_of_fame):
        if i < len(genomes):
            genomes[i] = genome.clone()

    print(
        f"Generation {g}: "
        f"best={max(fitness):.2f}, "
        f"avg={sum(fitness)/len(fitness):.2f}"
    )

Generation 0: best=933.31, avg=211.56
Generation 1: best=4397.66, avg=706.47
Generation 2: best=1803.19, avg=304.89
Generation 3: best=2546.06, avg=809.99
Generation 4: best=5130.19, avg=1147.12
Generation 5: best=1393.87, avg=316.50
Generation 6: best=2491.13, avg=513.15
Generation 7: best=4340.72, avg=797.20
Generation 8: best=5153.80, avg=1082.12
Generation 9: best=3761.44, avg=584.32
Generation 10: best=2674.03, avg=423.03
Generation 11: best=2187.22, avg=440.27
Generation 12: best=1668.61, avg=345.53
Generation 13: best=3132.60, avg=663.95
Generation 14: best=5489.56, avg=1737.24
Generation 15: best=2811.49, avg=485.52
Generation 16: best=3183.84, avg=706.00
Generation 17: best=2121.98, avg=508.47


KeyboardInterrupt: 

In [14]:


env = Environment()

agents = []

for genome in genomes:

    agent = Agent(genome)

    agent.x = torch.randint(0, env.width, (1,)).item()
    agent.y = torch.randint(0, env.height, (1,)).item()
    agent.theta = 2 * torch.pi * torch.rand(1).item()

    agents.append(agent)

frames = []

for t in range(500):

    env.update(agents)

    for agent in agents:
        agent.step(env, dt=1.0)

    frames.append((255 * env.field).byte().numpy())

iio.imwrite("simulation_v2.mp4", frames, fps=20)

In [15]:
from IPython.display import Video

Video("simulation_v2.mp4", embed=True)